In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('../auxiliary_data/kyrgyz-constitution-2021_corpus.csv')

In [3]:
from datasets import load_dataset
df_kir = load_dataset('cointegrated/panlex-meanings', name='kir', split='train').to_pandas()
df_eng = load_dataset('cointegrated/panlex-meanings', name='eng', split='train').to_pandas()

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
# creating kyr to eng dict
df_kir_eng = df_kir.merge(df_eng, on='meaning', suffixes=['_kir','_eng'])
kir_eng_pairs = df_kir_eng[['txt_kir','txt_eng']].drop_duplicates()

In [5]:
# creating eng to kyr dict
df_eng_to_kir = df_eng.merge(df_kir, on='meaning', suffixes=['_eng','_kir'])
eng_to_kir_pairs = df_eng_to_kir[['txt_eng','txt_kir']].drop_duplicates()

In [6]:
kyr_sentences = data['kyrgyz_tokenized13a']
eng_sentences = data['english_tokenized13a']

In [7]:
import json
# load data parsed by apertium
kyr_json = json.load(open('../auxiliary_data/apertium_constitution.json', 'r'))

In [8]:
from streamparser import LexicalUnit

def get_lex_units(units):
    """
        get from strings format that apertium returns
    """
    return [LexicalUnit(unit) for unit in units]

In [9]:
kyr_nouns = [get_lex_units(units) for units in kyr_json]

In [10]:
def parse_unit(unit):
    """
        iterate over apertiums analysis and extract all lemmas for one exact word(unit)
    """
    nouns = []
    for reading in unit.readings:
        for sread in reading:
            if 'n' in sread.tags:
                # extracting baseform of kyrgyz word
                nouns.append(sread.baseform)

    return {
        'surface': unit.wordform,
        'nouns': list(set(nouns)),
    }

In [11]:
def get_nouns_from_sentence(units):
    """
        iterate over units in one sentence and extract all lemmas which were found in sentence
    """
    noun_lemmas = []
    for i, unit in enumerate(units):
        analysis_result = parse_unit(unit)
        analysis_result['token_id'] = i
        if analysis_result['nouns']:
            noun_lemmas.append(analysis_result)
    return noun_lemmas

In [12]:
import re
def translate(words, vocab=kir_eng_pairs, lang_source='kir', lang_target='eng'):
    """
        get translations for lemmas extracted from one noun
    """
    matches_arrays = [vocab.loc[vocab[f"txt_{lang_source}"] == word, f"txt_{lang_target}"].unique() for word in words]
    # get set from translations, re.sub for russian pipeline only
    matches = {re.sub('\u0301', '', token) for arr in matches_arrays for token in arr }
    return list(matches)

In [19]:
def add_translations(token_dct):
    """
        Add translations to every noun in sentence
    """
    for unit in token_dct:
        unit['translations'] = translate(unit['nouns'])
    return token_dct

In [20]:
def eng_align(noun_translations, sentence_split):
    """
        noun_translations - list of translations of one kyrgyz word
        Iterate over result of eng parsing, if baseform of english word is in noun_translations, add to matches list
    """
    matches = []
    for index, eng_noun, eng_parse in sentence_split:
        if eng_parse.upos == 'NOUN':
            if eng_parse.lemma in noun_translations:
                matches.append((index, eng_noun, eng_parse.lemma))
    return matches

In [21]:
import stanza
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma')

2026-05-13 22:44:26 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-05-13 22:44:27 INFO: Downloaded file to /Users/mrdrozin/Library/Caches/stanza/1.11.0/resources/resources.json
2026-05-13 22:44:27 WARNING: Language en package default expects mwt, which has been added
2026-05-13 22:44:27 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2026-05-13 22:44:27 INFO: Using device: cpu
2026-05-13 22:44:27 INFO: Loading: tokenize
2026-05-13 22:44:28 INFO: Loading: mwt
2026-05-13 22:44:28 INFO: Loading: pos
2026-05-13 22:44:28 INFO: Loading: lemma
2026-05-13 22:44:28 INFO: Done loading processors!


В КОДЕ ИНДЕКС ТОКЕНА НА КЫРГЫЗСКОМ ЭТО РЕЗУЛЬТАТ ПАРСИНГА АПЕРТИУМА, ИНДЕКС НА РУССКОМ ЭТО РЕЗУЛЬТАТ СПЛИТА НА РУССКОМ

In [26]:
hard_cases = []
wsd_queries = []
all_wsd_queries = []
all_nouns = 0

In [29]:
def kyr_wsd(sentence_id: int, src_sentences, tgt_sentences):
    """
        Align noun pairs from source and target sentences by its translations
        Exclude cases of matching one-to-many and many-to-one
    """
    sentence_src = src_sentences[sentence_id]
    sentence_tgt = tgt_sentences[sentence_id]
    
    # get lemmas for all words and add translations to them
    nouns = get_nouns_from_sentence(sentence_src)
    nouns = add_translations(nouns)

    # split target sentence
    sent = nlp(sentence_tgt).sentences[0]
    sentence_split = [(index, analysis.text, analysis) for index, analysis in enumerate(sent.words)]
    
    counts = [0] * len(sentence_split)
    answers = []
    for unit in nouns:
        res = eng_align(unit['translations'], sentence_split)
        if res:
            global all_nouns
            all_nouns += 1
            eng_matched_lemmas = list({(ind, lemma) for ind, _, lemma in res})
            # case when one kyr word is matched to multiple english
            if len(eng_matched_lemmas) > 1:
                dct = {
                    'sentence_id': sentence_id,
                    'kyr_token_id': unit['token_id'],
                    'wordform': unit['surface'],
                    'lemmas': [lemma for _, lemma in eng_matched_lemmas],
                    'lemmas_ids': [ind for ind, _ in eng_matched_lemmas]
                }
                hard_cases.append(dct)
                continue
            #now we are sure that len(lemmas) = 1(it never equals 0, checked manually)
            matched_lemma = eng_matched_lemmas[0]
            dct = {
                'sentence_id': sentence_id,
                'kyr_token_id': unit['token_id'],
                'wordform': unit['surface'],
                'kyr_lemmas': unit['nouns'],
                'lemma': matched_lemma[1],
                'lemma_id': matched_lemma[0]
            }
            # count number of references to one english word to exclude many to one
            counts[matched_lemma[0]] += 1
            answers.append(dct)
            all_wsd_queries.append(dct)
    for dct in answers:
        # leave only one to one cases
        if counts[dct['lemma_id']] == 1:
            wsd_queries.append(dct)

In [30]:
for i in range(len(kyr_nouns)):
    kyr_wsd(i, kyr_nouns, eng_sentences)

In [31]:
all_nouns

1036

In [32]:
len(all_wsd_queries)

816

In [33]:
len(wsd_queries)

704

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
    ~~~~~~~~^^
  File "/opt/miniconda3/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 348, in dispatch_control
    await self.process_control(msg)
  File "/opt/miniconda3/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 354, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.13/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.resul

In [24]:
len(hard_cases)

219

In [25]:
from nltk.corpus import wordnet as wn

In [ ]:
final_dicts = []

# preparing data for API calls to DeepSeek
for query in wsd_queries:
    sentnence_id = query['sentence_id']
    lemma = query['lemma']
    lemma_id = query['lemma_id']
    sentnence = eng_sentences[sentnence_id].split()
    sentnence.insert(lemma_id + 1, '<WSD>')
    sentnence.insert(lemma_id, '<WSD>')
    synsets = wn.synsets(lemma)
    noun_synsets = [sns for sns in synsets if sns.pos() == 'n']
    synset_descriptions = ["; ".join([str(sns.name()), lemma, str(sns.definition())]) for sns in synsets]
    dct = {
        'sentence': " ".join(sentnence),
        'word': lemma,
        'descriptions': synset_descriptions
    }
    final_dicts.append(dct)

In [ ]:
with open('queries.json', 'r') as file:
    queries = json.load(file)
    
queries

In [ ]:
def get_cot_prompt(word: str, sentence: str, synset_descriptions):
    """
        Get prompt accordingly to paper about LLM WSD evaluation
    """
    parts = ['You are going to identify the corresponding sense tag of an ambiguity word in English sentences. Do the following tasks.', 
        f'1. {word} has different meanings. Below are possible meanings. Comprehend the sensetags and meanings.']
    parts.extend(synset_descriptions)
    parts.append('2. Now examine the sentence below. You are going to identify the most suitable meaning for ambiguity word.')
    parts.append(sentence)
    parts.extend([
        '3. Try to identify the meaning of the word in the above sentence which is enclosed with the <WSD>. You can think of the real meaning of \
sentence and decide the most suitable meaning for the word.',
        '4. Based on the identified meaning, try to find the most appropriate senseIDs from the below. You are given definition of each sense tag too.'
    ])
    parts.extend(synset_descriptions)
    parts.extend([
        '5. If you have more than one senseIDs identified after above steps, you can return the senseIDs in order of confidence level.',
        '6. Return JSON object that contains the ambiguity word and the finalized senseIDs. Use the following format for the output.',
        '<JSON Object with fields ambiguity_word and senseIDs >'
    ])
    return "\n".join(parts)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("DEEPSEEK")

client = OpenAI(
    api_key=key,
    base_url="https://api.deepseek.com",
)

In [ ]:
messages = [{"role": "system", "content": prompt}]

response = client.chat.completions.create(
    model="deepseek-reasoner",
    messages=messages,
    response_format={
        'type': 'json_object'
    }
)

print(json.loads(response.choices[0].message.content))

{'ambiguity_word': 'traditions', 'senseIDs': ['tradition.n.01', 'custom.n.02']}


In [112]:
json.loads(response.choices[0].message.content)

{'ambiguity_word': 'traditions', 'senseIDs': ['tradition.n.01', 'custom.n.02']}

In [ ]:
import tqdm

for dct in tqdm.tqdm(final_dicts):
    prompt = get_cot_prompt(dct['word'], dct['sentence'], dct['descriptions'])
    messages = [{"role": "system", "content": prompt}]
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=messages,
        response_format={
            'type': 'json_object'
        }
    )
    dct['annotation'] = json.loads(response.choices[0].message.content)

with open('deepseek_eng_results.json', 'w') as f:
    json.dump(final_dicts, f, indent=4)